# 温度采样（Temperature Sampling）

> 通过温度参数调节输出分布的"尖锐程度"，控制生成多样性。

## 背景
温度采样在 softmax 前除以温度 T：T<1 使分布更尖锐（趋近贪心），
T>1 使分布更平坦（增加多样性），T=1 为原始分布。

## 公式
$$P_T(y) = \text{softmax}\left(\frac{\text{logits}}{T}\right) = \frac{e^{z_y/T}}{\sum_j e^{z_j/T}}$$
- T → 0：趋近 argmax（贪心）
- T = 1：原始分布
- T → ∞：趋近均匀分布

## 复杂度
- 时间：O(V)，逐元素除法 + softmax
- 空间：O(V)
- 典型 T：0.7-1.2

## 考察点
- T<1 的效果：更确定、更保守，适合事实性任务
- T>1 的效果：更多样、更有创意，适合创意写作
- 与 Top-K/Top-P 组合：先调温度再截断，常用组合


In [ ]:
import torch
import torch.nn.functional as F

def temperature_sampling(logits: torch.Tensor, temperature: float = 1.0) -> torch.Tensor:
    """logits: [batch, vocab]，返回采样的 token id [batch]"""
    assert temperature > 0, "temperature must > 0"
    probs = F.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1).squeeze(-1)

In [ ]:
# 验证：不同温度下分布形状
torch.manual_seed(0)
logits = torch.tensor([[1.0, 2.0, 3.0, 4.0, 5.0]])
for T in [0.5, 1.0, 2.0, 10.0]:
    probs = F.softmax(logits / T, dim=-1)
    print(f'T={T:5}: {probs[0].tolist()}  最大概率={probs.max().item():.3f}')

# T→0 趋近 argmax
import math
probs_tiny = F.softmax(logits / 1e-3, dim=-1)
print('T→0 接近 one-hot(argmax=4):', probs_tiny[0].tolist())

## 小结 / 易错点
- 温度只改变分布**形状**，不截断；单独用温度仍可能采到低概率 token，故常配 TopK/TopP。
- $T$ 必须 $>0$；$T=0$ 要特判为 argmax（不能真除 0）。
- 顺序：温度在前，TopK/TopP 在后；先截断再调温度会改变截断语义。
- 原仓库版本已基本正确，本版补了原理、$T\to0$ 极限验证与组合顺序说明。

## ✅ 测试验证

In [ ]:
# 验证温度采样
import torch
import torch.nn.functional as F

logits = torch.randn(100)

# 温度采样: softmax(logits / T)
# T -> 0: 趋近贪心（one-hot）
# T = 1: 原始分布
# T -> inf: 趋近均匀分布

# T=1: 与原始 softmax 一致
probs_t1 = F.softmax(logits / 1.0, dim=-1)
probs_ref = F.softmax(logits, dim=-1)
assert torch.allclose(probs_t1, probs_ref, atol=1e-6), "T=1 should match original"

# T 很小: 趋近 one-hot（最大概率趋近 1）
probs_cold = F.softmax(logits / 0.01, dim=-1)
max_prob_cold = probs_cold.max().item()
assert max_prob_cold > 0.99, f"cold temp should be near one-hot, max={max_prob_cold}"

# T 很大: 趋近均匀
probs_hot = F.softmax(logits / 100.0, dim=-1)
assert probs_hot.std().item() < 0.01, f"hot temp should be near uniform, std={probs_hot.std()}"

# 验证: 温度改变不改变 argmax（最可能 token 不变）
assert logits.argmax() == (logits / 0.5).argmax() == (logits / 2.0).argmax(), \
    "temperature should not change argmax"

print("✅ TemperatureSampling 测试通过: T=1原始, T→0贪心, T→∞均匀, argmax不变")
